# CNN Poll of Polls
> This notebook fetches and processes the network's [time-series of aggregates](https://www.cnn.com/polling/approval/trump-cnn-poll-of-polls). 

---

#### Import Python tools and Jupyter config

In [51]:
import us
import json
import requests
import pandas as pd
import jupyter_black

In [52]:
jupyter_black.load()
pd.options.display.max_columns = 100
pd.options.display.max_rows = 100
pd.options.display.max_colwidth = None

---

In [53]:
def extract_value(questions, label):
    return next((q["value"] for q in questions if q["label"] == label), None)

In [54]:
headers = {
    "accept": "*/*",
    "accept-language": "en-US,en;q=0.9,es;q=0.8",
    "origin": "https://www.cnn.com",
    "referer": "https://www.cnn.com/",
    "user-agent": "Mozilla/5.0",
}

In [55]:
# Polling url
poll_of_polls_url = (
    "https://politics.api.cnn.io/polling/trump-second-term-approval/poll-of-polls.json"
)

In [56]:
# Make request
response = requests.get(
    poll_of_polls_url,
    headers=headers,
)

data = response.json()

In [57]:
# Flatten the aggregate series
agg_rows = []

for snap in data:
    agg_rows.append(
        {
            "timestamp": pd.to_datetime(snap["timestamp"], unit="s"),
            "start_date": pd.to_datetime(snap["start_date"], unit="s"),
            "end_date": pd.to_datetime(snap["end_date"], unit="s"),
            "approve": extract_value(snap["questions"], "Approve"),
            "disapprove": extract_value(snap["questions"], "Disapprove"),
            "net": (
                extract_value(snap["questions"], "Approve")
                - extract_value(snap["questions"], "Disapprove")
            ),
            "n_polls": len(snap["entries"]),
        }
    )

df_agg = pd.DataFrame(agg_rows).sort_values("timestamp")

In [59]:
## Flatten the underlying polls
poll_rows = []

for snap in data:
    snap_ts = pd.to_datetime(snap["timestamp"], unit="s")

    for p in snap["entries"]:
        poll_rows.append(
            {
                "snapshot_ts": snap_ts,
                "pollster": p["pollster"],
                "start_date": pd.to_datetime(p["start_date"], unit="s"),
                "end_date": pd.to_datetime(p["end_date"], unit="s"),
                "release": pd.to_datetime(p["release"], unit="s"),
                "approve": extract_value(p["questions"], "Approve"),
                "disapprove": extract_value(p["questions"], "Disapprove"),
                "net": (
                    extract_value(p["questions"], "Approve")
                    - extract_value(p["questions"], "Disapprove")
                ),
                "moe": p.get("moe"),
                "sample_size": p.get("sample_size"),
                "sample_type": p.get("sample_type"),
                "source_uri": p.get("source_uri"),
            }
        )

df_polls = pd.DataFrame(poll_rows)

---

## Exports

#### XyXy subset in CSV format to `processed`

#### JSON, GeoJSON, etc., to `processed`